# Sprint 3 · An observable FieldCare service — Campus

HelioDesk technicians need visible progress while they wait for a longer explanation. Dispatch still needs a complete machine-readable response. You will preserve your protected service, compare buffered and incremental delivery, inspect safe operational records, and use a signal to revisit the **existing Module A evaluation cases**.

**Keep this one notebook for the entire sprint.** Save a copy in Drive before editing. Run sections in order; return to the same sections as you work through Campus. Keep all source changes in the printed workspace folder. Later cells extend that same service.

| Section | What you build or decide |
|---|---|
| 1 · Start | Restore your secured Sprint 2 checkpoint, or use labelled recovery |
| 2 · Choose delivery | Match streaming to the consumer's task |
| 3 · Stream | Register incremental delivery; observe completion, failure and cancellation |
| 4 · Read safe signals | Connect request identifiers, latency, usage and outcomes |
| 5 · Minimize and check | Exercise synthetic private markers and retained safe metadata |
| 6 · Close the loop | Run the original evaluator on the service's configuration |
| 7 · Checkpoint | Investigate a new shift request mix independently |
| 8 · Save | Export one source/evidence checkpoint and save this notebook |

The default is an explicitly scripted **transport fixture**: no model call, bill or measured LLM speed. The optional live branch uses real incremental provider generation. Neither a fixture nor HTTP 200 proves answer quality. The original evaluator tests deterministic evidence, tool use and decision flags; live wording needs separate review.

Use this Campus notebook for every Campus lesson in this sprint. The live sessions use a separate Live notebook. Save one personal copy of each; keep your cumulative project here.


## Campus lessons in Notion

- [When streaming helps](https://app.notion.com/p/3ea5a55972ad81fd9cdee92887860939?pvs=204)
- [Enabling streaming on the scaffold](https://app.notion.com/p/3ea5a55972ad8118a84ef55a5d48442e?pvs=204)
- [What to log and why](https://app.notion.com/p/3ea5a55972ad817f865ec69c602dd137?pvs=204)
- [What NOT to log](https://app.notion.com/p/3ea5a55972ad8168bb14fe42cd0c288f?pvs=204)
- [Closing the loop with existing evaluation](https://app.notion.com/p/3ea5a55972ad816f89b7f0bd61f6d601?pvs=204)
- [Observable streaming service checkpoint](https://app.notion.com/p/3ea5a55972ad81d9b6dafb3e76ff75fd?pvs=204)


## 1 · Start with your secured service

Run the setup once. In Colab the published course GitHub URL and pinned revision load the shared library; a local checkout is reused. No supporting source ZIP is needed. Installation errors should be fixed before continuing; restarting a session does not recover unsaved local files.

Set `SPRINT2_CHECKPOINT` to your personal export's path. Upload that personal checkpoint through Colab's Files panel if needed. This is your saved work, not a course-source archive. `USE_DEMO_RECOVERY=True` allows you to rehearse when your earlier work is unavailable, but its source is not evidence that you completed the prerequisite.

To resume this sprint, set `RESUME_ZIP` to your most recent personal export and keep the same `RUN_NAME`. Rerunning setup never replaces existing workspace files. To start another attempt, choose a new name.

In [ ]:
#@title Load the shared course code from GitHub
from pathlib import Path
import os, re, sys, subprocess, tempfile

# Reviewed shared helpers: this immutable revision keeps every lesson on the same source.
GITHUB_REPO = 'https://github.com/richhiey/ai-app-dev_Mod-B.git'
GITHUB_REF = 'fb9d95b1cadcc29972cb9dd7388201aaa5868e1d'
repo_url = os.environ.get('MODULE_B_REPO_URL', GITHUB_REPO).strip()
repo_ref = os.environ.get('MODULE_B_REPO_REF', GITHUB_REF).strip()
destination = Path(os.environ.get('MODULE_B_REPO', '/content/ai-app-dev_Mod-B')).expanduser()
candidates = [destination, Path.cwd(), *Path.cwd().resolve().parents]
REPO = next((p.resolve() for p in candidates if (p/'src/module_b/notebook.py').is_file()), None)

if REPO is None:
    if not re.fullmatch(r'https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?', repo_url):
        raise RuntimeError('The course author must set GITHUB_REPO to the published HTTPS GitHub repository URL.')
    if not re.fullmatch(r'[0-9a-fA-F]{40}|refs/tags/[A-Za-z0-9][A-Za-z0-9._/-]*', repo_ref):
        raise RuntimeError('The course author must set GITHUB_REF to a release tag (refs/tags/...) or full commit SHA.')
    if destination.exists():
        raise FileExistsError('The destination already exists without the expected helpers. Preserve it and use a fresh runtime or a new MODULE_B_REPO path.')
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Prepare separately, so a failed download cannot replace existing student work.
    with tempfile.TemporaryDirectory(prefix='module-b-download-', dir=destination.parent) as temporary:
        checkout = Path(temporary)/'checkout'
        subprocess.run(['git','clone','--no-checkout','--depth','1',repo_url,str(checkout)],check=True)
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin',repo_ref],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
        if not (checkout/'src/module_b/notebook.py').is_file() or not (checkout/'requirements.lock').is_file():
            raise RuntimeError('The selected repository revision does not contain the expected Module B source layout.')
        checkout.rename(destination)
    REPO = destination.resolve()

# Existing local checkouts are reused; setup never pulls, resets or overwrites them.
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'-c',str(REPO/'requirements.lock')],check=True)
if str(REPO/'src') not in sys.path:
    sys.path.insert(0,str(REPO/'src'))
from module_b import __version__
print('Shared package ready:', __version__)
print('Source checkout:', REPO)

In [ ]:
import json, time, tempfile, copy
import httpx
from module_b.evaluation import module_a, load_pipeline, design_revision, evaluate_selected
from module_b.observability import FIELDS, safe_record
from module_b.observability_lab import (prepare_observable_workspace, save_guided_file,
    insert_before_security, service, collect_stream, read_logs, probe_checkpoint, review_checkpoint)
from module_b.notebook import table, register_source, revision, export_progress
from module_b.workspace import restore_workspace

In [ ]:
RUN_NAME = 'sprint-3'
SPRINT2_CHECKPOINT = ''
USE_DEMO_RECOVERY = True
RESUME_ZIP = ''
work = REPO/'work'/RUN_NAME
if RESUME_ZIP and not (work/'service').exists():
    restore_workspace(RESUME_ZIP, work/'service')
if not SPRINT2_CHECKPOINT and not USE_DEMO_RECOVERY and not RESUME_ZIP:
    raise ValueError('Select your checkpoint, or explicitly choose demo recovery.')
project, inherited = prepare_observable_workspace(work/'service',
    checkpoint=SPRINT2_CHECKPOINT or None, repo_root=REPO)
patterns = REPO/'examples/patterns/sprint_3'
evidence = {}
print(inherited['provenance'])
print('Your cumulative source:', project)

## 2 · Choose delivery for the consumer

**Streaming** sends a response incrementally while work continues. A technician reading an explanation can begin before it finishes. A dispatch system that must validate a complete JSON object cannot safely act on half an object. Retain its buffered contract.

**First-content time** measures the wait until useful content becomes visible; **completion time** measures the wait until the terminal result. A shorter first-content time does not imply shorter total time or better content. A transport chunk is neither a complete answer nor necessarily a model token.

Write a decision for each consumer: choose a delivery mode and define what counts as successful completion. Include the behavior after partial content followed by failure. Leave these fields empty until you have a reasoned answer.

In [ ]:
delivery_decisions = {'technician': '', 'dispatch': '', 'partial_failure': ''}

## 3 · Enable and observe streaming

The supplied bridge reuses the broader Module A FieldCare application using approved synthetic case identifiers. It adds replay endpoints; it does not change the narrower existing `/v1/diagnose` contract. `case_id` selects an original evaluation fixture. An optional `question` replaces that fixture's text for a controlled experiment; the tag alone does not prove it is still the approved case.

**Action:** copy the reusable event engine once, then register the routes below. **Reason:** the same event source makes buffered and streamed delivery comparable. `yield` hands one encoded event to the response as it becomes available. New routes register **before** the existing guard inventories POST paths. The observation layer is placed outside the guard so rejections receive a server-generated correlation identifier too.

**Expected output:** paths to your saved files. An existing file is kept on rerun. Inspect it before editing if your implementation differs from this worked example.

The protocol is **NDJSON**: one complete JSON object per line. A `delta` carries text; `complete` marks success; `error` marks failure. HTTP status is committed before the whole stream arrives. Read lines as they arrive instead of calling `.json()` on the full streaming response. See [HTTPX streaming responses](https://www.python-httpx.org/quickstart/#streaming-responses).

In [ ]:
# Engine mechanics live in shared source; route and yield remain visible here.
source = (patterns/'stream_routes.py').read_text()
engine = source.split("@router.post('/v1/fieldcare-stream')")[0]
save_guided_file(project, 'app/stream_engine.py', engine)
routes = """from fastapi import APIRouter, Request
from fastapi.responses import StreamingResponse
from app.stream_engine import ReplayRequest, events, pipeline, DESIGN, PIPELINE_REVISION
from module_b.streaming import encode_event
from module_b.evaluation import run_case, design_revision
router = APIRouter()

@router.post('/v1/fieldcare-stream')
async def stream(body: ReplayRequest, request: Request):
    async def lines():
        async for event in events(body, request):
            yield encode_event(event)
    return StreamingResponse(lines(), media_type='application/x-ndjson',
                             headers={'Cache-Control': 'no-store'})

@router.post('/v1/fieldcare-buffered')
async def buffered(body: ReplayRequest, request: Request):
    return {'events': [event async for event in events(body, request)]}

@router.post('/v1/fieldcare-evidence')
async def evidence(body: ReplayRequest, request: Request):
    context, response = run_case(pipeline, body.case_id, body.question)
    request.state.observation.update(source='module_a_deterministic',
                                    model='module-a-rules', case_id=body.case_id)
    return {'response': response, 'design_revision': PIPELINE_REVISION}
"""
save_guided_file(project, 'app/stream_routes.py', routes)
# This exact file is read by BOTH the service and the original evaluator.
design_path = save_guided_file(project, 'data/pipeline_design.json',
    json.dumps(module_a.default_pipeline_design(), indent=2))
registration = '\nfrom app.stream_routes import router as stream_router\napp.include_router(stream_router)\n'
insert_before_security(project, registration)
# Safe minimal logging is installed now; inspect its fields in the next section.
log_source = "LOG_FIELDS = " + repr(FIELDS) + '\n'
save_guided_file(project, 'app/log_settings.py', log_source)
attachment = (patterns/'attachment.py.txt').read_text()
print(attachment)
register_source(project, attachment)
print('Saved routes:', project/'app/stream_routes.py')

### Compare the same request through real local HTTP

**Action:** run the comparison. Each process owns an isolated set of ephemeral keys and counters; keys are kept in memory and never printed. **Reason:** a fresh controlled runtime avoids interpreting an earlier exhausted allowance as a streaming failure. This restart is a lab control, not a production limit workaround.

**Expected:** the fixture's first `delta` arrives before its `complete` event. Buffered and streamed text match, while correlation identifiers differ. Timings vary by machine and include local overhead. A lower first-content time is evidence about this transport fixture only.

**Check:** both successful sequences end with `complete`. The interrupted sequence ends with `error`, even if its HTTP status is 200. Your code must not promote partial text to success.

In [ ]:
# Each delivery observation gets a fresh worker; no quota-renewal claim follows.
guided_logs = []
for mode in ['buffered', 'normal', 'interrupted']:
    with tempfile.TemporaryDirectory() as temporary:
        log = Path(temporary)/'requests.jsonl'; server, keys = service(project, log)
        with server, httpx.Client(base_url=server.base_url, trust_env=False, timeout=20) as client:
            headers = {'X-API-Key': next(iter(keys.values()))}
            payload = {'case_id': 'EVAL-FC-005'}
            start = time.perf_counter()
            if mode == 'buffered':
                buffered = client.post('/v1/fieldcare-buffered', json=payload, headers=headers)
                buffered_ms = (time.perf_counter()-start)*1000
            else:
                if mode == 'interrupted': payload['fault'] = 'interrupt'
                with client.stream('POST', '/v1/fieldcare-stream', json=payload, headers=headers) as response:
                    observation = collect_stream(response, start)
                if mode == 'normal': normal = observation
                else: interrupted = observation
            guided_logs.extend(read_logs(log, expected=1))
same_text = ([e.get('text') for e in buffered.json()['events']] == [e.get('text') for e in normal['events']])
assert same_text and normal['completed'] and not interrupted['completed']
assert normal['first_content_ms'] < normal['completion_ms']
table([{'delivery':'buffered','first_content_ms':round(buffered_ms,1),'completion_ms':round(buffered_ms,1),'terminal':'complete'},
       {'delivery':'streamed','first_content_ms':round(normal['first_content_ms'],1),'completion_ms':round(normal['completion_ms'],1),'terminal':normal['event_types'][-1]},
       {'delivery':'interrupted','status':interrupted['status_code'],'terminal':interrupted['event_types'][-1]}])
evidence['comparison'] = {'same_fixture_text':same_text,'experiments':'three fresh delivery workers; no renewal claim',
    'normal':{k:v for k,v in normal.items() if k!='events'},
    'interrupted':{k:v for k,v in interrupted.items() if k!='events'}}


### Distinguish interruption from cancellation

A provider interruption fails while the client is still waiting. **Cancellation** is the client closing its stream before completion. Close after the first fixture line and inspect the terminal record. Expected: `cancelled` if the disconnect reaches the server before it completes. A fast response can finish first; that is a race, not proof that cancellation never works. The runtime context always stops the owned process.

The live adapter closes its upstream connection during cleanup. Provider billing/cancellation behavior varies; do not infer that a client disconnect always stops billed generation. The [provider streaming documentation](https://openrouter.ai/docs/api_reference/streaming) describes that limitation.

In [ ]:
with tempfile.TemporaryDirectory() as temporary:
    log = Path(temporary)/'requests.jsonl'
    server, keys = service(project, log)
    with server, httpx.Client(base_url=server.base_url, trust_env=False) as client:
        headers = {'X-API-Key': next(iter(keys.values()))}
        with client.stream('POST', '/v1/fieldcare-stream',
                           json={'case_id': 'EVAL-FC-006'}, headers=headers) as response:
            for line in response.iter_lines():
                if line:
                    break  # Closing the context releases this connection.
        cancellation_logs = read_logs(log, expected=1)
    table(cancellation_logs, ['request_id', 'outcome', 'error_category', 'latency_ms'])
evidence['cancellation'] = cancellation_logs

### Optional · Observe actual incremental generation

The engine sends `stream=True` to the provider, reads incoming events and forwards content. It does not split a completed answer into pretend model chunks. Set runtime secrets using Colab Secrets or environment variables without printing values. Set `LIVE_MODEL` to an allowed course model. The optional cell reads an existing runtime key or Colab Secret and otherwise asks through hidden input. Run this optional cell only when you intentionally want a provider request; it can incur usage.

The fixture interruption switch is for local rehearsal only. Real provider errors are mapped to fixed safe categories. The adapter requires a clean provider stop and terminator; truncation is not accepted as completion. Live generated wording is **not** graded by the deterministic evaluator later in this notebook.

In [ ]:
RUN_LIVE = False
LIVE_MODEL = ''  # Set the course-approved model identifier before opting in.
if RUN_LIVE:
    if not LIVE_MODEL.strip():
        raise ValueError('Set LIVE_MODEL before making a provider request.')
    if not os.environ.get('OPENROUTER_API_KEY'):
        try:
            from google.colab import userdata
            os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY') or ''
        except Exception:
            pass
        if not os.environ.get('OPENROUTER_API_KEY'):
            from getpass import getpass
            os.environ['OPENROUTER_API_KEY'] = getpass('Provider key (hidden, runtime only): ').strip()
    os.environ['OPENROUTER_MODEL'] = LIVE_MODEL.strip()
    with tempfile.TemporaryDirectory() as temporary:
        log = Path(temporary)/'requests.jsonl'
        server, keys = service(project, log, mode='live')
        with server, httpx.Client(base_url=server.base_url, trust_env=False, timeout=60) as client:
            start = time.perf_counter()
            with client.stream('POST', '/v1/fieldcare-stream', json={'case_id': 'EVAL-FC-001'},
                headers={'X-API-Key': next(iter(keys.values()))}) as response:
                live = collect_stream(response, start)
            live['evidence_kind'] = 'actual_local_http_live_provider'
            print('Terminal state:', live['event_types'][-1] if live['event_types'] else 'no events')
            table(read_logs(log, expected=1))
else:
    print('Live provider branch not run. No generated-answer evidence claimed.')

## 4 · Read safe operational signals

A **structured log** records named fields rather than an arbitrary message. A **correlation identifier** joins observations for one runtime request. Our `request_id` is generated by the server and returned in `X-Request-ID`; an incoming header cannot choose it. It is distinct from Module A's fixture `request_id` and evaluation `eval_id`.

| Field | Question it helps answer | Limit |
|---|---|---|
| `request_id`, `route` | Which attempt and operation? | A unique identifier is not a user's identity |
| `source`, `model` | Fixture, rules or provider? Which approved model was reported? | Unknown model is `null`; no invented model name |
| `tokens` | What usage count did the provider report? | `null` means unknown, never zero-cost |
| `first_content_ms`, `latency_ms` | First progress and total server work? | Server measurements differ from client/network timing |
| `status_code`, `outcome`, `error_category` | Rejected, completed, failed or cancelled? | HTTP 200 is not final stream success |
| `case_id` | Which approved synthetic case was selected? | A candidate for review, not proof of case equivalence |

The server records `first_content_ms` when its event engine produces the first content. A buffered route can therefore have a small server value while the client still waits for the full object. Use the client comparison to establish visible progress.

**Action:** inspect the explicit allowlist and correlate the comparison's IDs with its logs. **Reason:** one terminal record per request prevents counting each chunk as a successful request. **Expected:** the interrupted request has status 200, outcome `failed` and a safe category. Fixture usage is unknown.

The application logger retains none of the raw body, prompt, answer, headers or exception message. The owned runtime disables access logging and discards process output. When deploying elsewhere, review proxy/platform logs separately; this notebook cannot establish their behavior.

In [ ]:
print((project/'app/log_settings.py').read_text())
table(guided_logs)
assert len({r['request_id'] for r in guided_logs}) == 3
assert any(r['request_id']==normal['request_id'] and r['outcome']=='completed' for r in guided_logs)
assert any(r['request_id']==interrupted['request_id'] and r['outcome']=='failed' for r in guided_logs)
assert all(r['tokens'] is None for r in guided_logs)
signal_note = ''  # Explain the interrupted record and the limit of the token information.

### Interpret available usage without fabricating a measurement

The record below is explicitly a **provider-mock interpretation fixture**. It shows how an available usage value differs from the unknown value in your transport logs. It is not a runtime measurement, price estimate or evidence of live generation. Actual adapter behavior is separately covered by mocked provider tests in the repository.

In [ ]:
usage_examples = [
    {'evidence_kind': 'provider_mock_interpretation_fixture', 'tokens': 17, 'meaning': 'reported total'},
    {'evidence_kind': 'transport_fixture', 'tokens': None, 'meaning': 'unknown'}]
table(usage_examples)

## 5 · Minimize first; verify both exclusion and usefulness

**Redaction** removes or masks sensitive content before retention. **Allowlisting** selects only approved fields. Here we first exclude payloads entirely, then validate values of the fields we retain. A key named `model` is not permission to store an arbitrary string.

**Action:** run the deliberately unsafe synthetic input below through `safe_record`, then exercise the HTTP paths. **Reason:** a normal request alone misses validation and rejection paths. Include nested invalid data, query text, rejected credentials and a mid-stream failure. All markers are fake. Do not use real personal data or secrets.

**Expected:** the marker is absent, but IDs, routes and terminal outcomes still exist. Empty logs do not pass. The checks establish these supplied paths and marker patterns; they are not universal personal-data detection.

The helper runs two **separately labelled experiments**. The first keeps one process running while it checks every registered POST route (including routes hidden from OpenAPI), missing and wrong credentials, four admitted attempts, all three replay routes after exhaustion, and the other caller's inherited routes. The second starts a fresh process for buffered equivalence, client cancellation and a successful request afterward. This restart isolates delivery behavior; it is not evidence of timed allowance renewal.

For the deliberately delayed fixture, the client must observe at least 40 ms between its first content and completion. The fixture separates pieces by 80 ms, so this tolerant check detects accidentally buffered delivery. It is a local transport check, not an LLM speed target. If it fails, inspect client timings and event consumption before changing your service.

The probe returns metadata only. Its temporary logs are removed at exit; the marker itself is not included in exported evidence. The separate `unsafe_fixture` remains an intentionally unsafe dictionary in memory: creating `minimized` does not clean an existing object or an already-written file.

In [ ]:
import uuid
MARKER = 'SYNTHETIC-PRIVATE-person@example.invalid'
unsafe_fixture = {'request_id': str(uuid.uuid4()), 'route': '/v1/fieldcare-stream',
    'outcome': 'failed', 'source': 'transport_fixture', 'tokens': None,
    'prompt': MARKER, 'headers': {'X-API-Key': MARKER},
    'exception': {'message': MARKER}, 'model': MARKER, 'error_category': MARKER}
minimized = safe_record(unsafe_fixture, routes=('/v1/fieldcare-stream',))
assert MARKER not in json.dumps(minimized)
privacy_report = probe_checkpoint(project, case_id='EVAL-FC-005',
    companion='EVAL-FC-003', marker=MARKER)
assert privacy_report['marker_absent'] and privacy_report['safe_metadata_present']
assert privacy_report['unauthorized_status']==401 and privacy_report['limited_status']==429
print('Separate experiments:', privacy_report['experiments'])
table(privacy_report['logs'], ['request_id','route','status_code','outcome','error_category'])
evidence['privacy'] = privacy_report
privacy_note = ''  # What paths did you test, and what remains outside that evidence?

## 6 · Close the loop with the existing evaluation

A failed transport record points to delivery. A repeated application concern points to a candidate behavior to investigate. Neither is an answer-quality score. A **regression** is previously acceptable behavior that becomes incorrect after a change.

For this worked example, inspect the response to `EVAL-FC-005` (missing context) using the same configuration file the service imports. Select `EVAL-FC-003` as a companion because an equipment-evidence change can affect the HX/VX distinction. `EVAL-FC-006` covers unavailable warranty information. Read the original case rather than assuming what a tag means.

**Action:** load the application pipeline from `design_path`, read the original criteria, then call **Module A's actual `evaluate_case`**. **Reason:** evaluating an unrelated default pipeline would tell you nothing about your service. **Expected:** baseline cases pass; retain that honest result. Do not force an improvement when none is needed.

The evaluator compares required/forbidden documents, required tool calls and expected response flags. It does not grade streaming latency, confidentiality, or the truthfulness of optional live prose. The source snapshot and import-only adaptations are documented in `src/module_b/_module_a/provenance.json`.

In [ ]:
pipeline = load_pipeline(design_path)
GUIDED_CASES = ['EVAL-FC-005', 'EVAL-FC-003']
criteria = [module_a.eval_case_by_id(pipeline.env, case_id) for case_id in GUIDED_CASES]
table(criteria, ['eval_id','request_id','required_doc_ids','required_tool_calls','expected_response_flags','must_not_use_doc_ids'])
# The original evaluator is visible here; no new score is introduced.
guided_eval = [module_a.evaluate_case(pipeline, case) for case in criteria]
table(guided_eval, ['eval_id','pipeline_pass','missing_required_docs','missing_required_tools','missing_expected_flags'])
assert design_revision(design_path) == privacy_report['design_revision']
evidence['guided_evaluation'] = {'design_revision': design_revision(design_path), 'rows': guided_eval}

### Diagnose an intentional regression, then recover

The next experiment is deliberately broken: a **temporary copy** disables the rule that asks for equipment context before model-specific guidance. It does not alter your cumulative service. Predict the result for both cases before running it.

**Expected:** the missing-context case fails while the HX/VX companion remains a separate check. Restore the exact saved design and rerun both. A recovered result must come from a new evaluator run, not a renamed old table. This models the workflow: signal → approved case → criteria → narrow change → selected and companion reruns.

In [ ]:
regression_prediction = ''
with tempfile.TemporaryDirectory() as temporary:
    experiment = Path(temporary)/'design.json'
    original_design = design_path.read_text()
    broken = json.loads(original_design)
    broken['response_policy']['ask_before_model_specific_guidance_when_ids_missing'] = False
    experiment.write_text(json.dumps(broken))
    broken_eval = evaluate_selected(experiment, GUIDED_CASES)
    table(broken_eval['rows'], ['eval_id','pipeline_pass','missing_expected_flags'])
    experiment.write_text(original_design)
    recovered_eval = evaluate_selected(experiment, GUIDED_CASES)
    table(recovered_eval['rows'], ['eval_id','pipeline_pass','missing_expected_flags'])
eval_note = ''  # State what changed, what did not, and what the evaluator cannot establish.

## 7 · Observable streaming service checkpoint

A new HelioDesk shift handles an unavailable-warranty request and an equipment-specific evidence request. Technicians need progress; dispatch still needs complete objects. Reuse your existing notebook and workspace.

Produce evidence for a **fresh selected/companion case pair**, including `EVAL-FC-006`, and choose a new synthetic marker unlike the worked one. Read each original case first. Inspect a selected response against its original criteria, retain or make one justified configuration change, and rerun both cases. A justified no-change decision is acceptable when the criteria already pass.

Your submission must show successful progress and completion, interrupted failure, retained authorization/limit behavior and buffered contracts, useful logs without your private markers, and the original evaluation results connected to the configuration actually served. Explain the signal-to-case mapping: do not feed raw production logs into the evaluator.

Edit `app/stream_routes.py`, `app/log_settings.py`, or `data/pipeline_design.json` only if your investigation warrants it. Keep inherited source intact. After any edit restart through the probe and rerun evaluation; stale observations are not current evidence. The readiness check verifies evidence structure, not the quality of your reasoning. An instructor reviews the written judgments.

Leave `ASSESSMENT_READY=False` until you have chosen your experiment. Running all cells in an untouched notebook must remain **pending**, not become a completed assessment.

In [ ]:
ASSESSMENT_READY = False
selected_case = ''
companion_case = ''
assessment_marker = ''
notes = {'stream_decision': '', 'log_fields': '', 'privacy': '', 'eval_reason': '', 'result': ''}
assessment_before = {}; assessment_after = {}; assessment_observations = {}
if ASSESSMENT_READY:
    assert 'EVAL-FC-006' in {selected_case, companion_case}
    assert selected_case != companion_case
    assert assessment_marker.startswith('SYNTHETIC-') and assessment_marker != MARKER
    # Persist the actual entry measurement once for this chosen pair; reruns keep it.
    before_path = project/'fixtures'/f'eval-before-{selected_case}-{companion_case}.json'
    if not before_path.exists():
        before_path.write_text(json.dumps(evaluate_selected(design_path,[selected_case,companion_case]),indent=2))
    assessment_before = json.loads(before_path.read_text())
    table(assessment_before['rows'])
    selected_trace = module_a.run_eval_case(load_pipeline(design_path), selected_case)
    table([selected_trace['eval_case']], ['eval_id','required_doc_ids','required_tool_calls','expected_response_flags'])
    table([selected_trace['response']], ['answer_summary','decision_flags','escalation_path','limitations'])
else:
    print('Checkpoint pending: choose a case pair, marker and evidence plan.')

### Make or retain a justified design choice

Record the selected response and criteria in your own words. If you change configuration, put the exact complete updated design into `assessment_design`; leaving it `None` retains the current design. A missing change is not itself a failure: your evidence must justify the choice. Do not edit original fixture inputs or expected outcomes to make a test pass.

In [ ]:
assessment_design = None
if ASSESSMENT_READY and assessment_design is not None:
    # Validate with the original builder before changing the shared service file.
    module_a.build_fieldcare_pipeline(module_a.load_fieldcare_environment(), assessment_design)
    design_path.write_text(json.dumps(assessment_design, indent=2))

In [ ]:
if ASSESSMENT_READY:
    assessment_observations = probe_checkpoint(project, case_id=selected_case,
        companion=companion_case, marker=assessment_marker)
    assessment_after = evaluate_selected(design_path,[selected_case,companion_case])
    table(assessment_after['rows'], ['eval_id','pipeline_pass','missing_required_docs','missing_expected_flags'])
    print('Separate experiments:', assessment_observations['experiments'])
    table(assessment_observations['logs'], ['request_id','route','status_code','outcome','error_category'])
    evidence['assessment'] = {'observations': assessment_observations,
                            'before': assessment_before, 'after': assessment_after}
readiness = review_checkpoint(project, inherited=inherited,
    observations=assessment_observations, eval_before=assessment_before, eval_after=assessment_after,
    selected=selected_case, companion=companion_case, notes=notes)
print(readiness)

## 8 · Save your source and evidence

Run the final export once after your checks, and rerun it after any later change. The ZIP contains your permitted source, configuration and safe evidence. It is a **personal checkpoint** for the next sprint, not a supporting course-source bundle. Save this notebook separately in Drive and download the checkpoint before the temporary runtime ends.

If readiness is false, the export still saves your progress and explicitly records that status. Written responses remain pending when empty. Local automated checks do not certify a hosted Colab or deployed service run.

In [ ]:
assessment_observations = globals().get('assessment_observations', {})
assessment_before = globals().get('assessment_before', {})
assessment_after = globals().get('assessment_after', {})
selected_case = globals().get('selected_case', '')
companion_case = globals().get('companion_case', '')
notes = globals().get('notes', {k:'' for k in ['stream_decision','log_fields','privacy','eval_reason','result']})
readiness = review_checkpoint(project, inherited=inherited,
    observations=assessment_observations, eval_before=assessment_before, eval_after=assessment_after,
    selected=selected_case, companion=companion_case, notes=notes)
from module_b.live_sessions import read_records
evidence['live_workshops'] = read_records(project)
SPRINT3_EXPORT = export_progress(project, REPO/'artifacts', sprint=3, evidence=evidence,
    notes={**notes, 'delivery': globals().get('delivery_decisions', {}),
           'signals': globals().get('signal_note', ''),
           'guided_privacy': globals().get('privacy_note', ''),
           'guided_evaluation': globals().get('eval_note', '')}, readiness=readiness)
print('Readiness:', readiness)


## Quick reference

| Operation | Purpose |
|---|---|
| `StreamingResponse` + `yield encode_event(event)` | Incremental NDJSON delivery |
| `client.stream(...)` + `iter_lines()` | Observe client-visible arrivals |
| `safe_record(...)` | Keep only allowed metadata and value types |
| `X-Request-ID` | Correlate this runtime request with its terminal record |
| `module_a.evaluate_case(pipeline, case)` | Reuse the original behavior checks |
| `evaluate_selected(design_path, case_ids)` | Repeat those checks on a saved service design |
| `export_progress(...)` | Save one cumulative source/evidence checkpoint |

## Watch out for

- If the server does not start, check that your caller configuration has two labels, distinct runtime keys and the inherited four-attempt/60-second policy. Do not print keys to debug it.
- Registering routes after the security inventory can leave them uncovered; use the demonstrated insertion point and recheck every POST path.
- A terminal `error`, truncation or client close is not completion. A 200 response can still fail later.
- If logs are empty, check the observation attachment and path before celebrating a privacy pass.
- A `null` token value is unknown. A provider-mock count is not your usage.
- A changed design needs a fresh service process and a fresh evaluator run; compare configuration revisions.
- Do not copy a raw request, provider exception or secret into a notebook note, output, screenshot or export.

## Summary

Your service can now expose progress without discarding complete-response contracts. Each attempt leaves a useful bounded record, and a suspected application problem can be traced to the original case and criteria. The saved checkpoint carries this observable service into the interface work.